# JAX-GCM (JCM) Demo 

<img src="../logo.png" alt="JAX-GCM Logo" width="200"/>

This notebook demonstrates the capabilities of JAX-GCM (JCM), a differentiable general circulation model built using JAX. 

We will showcase how to set up a simple atmospheric simulation, run the model, and visualize the results. Additionally, we will illustrate how to compute gradients with respect to model parameters using JAX's automatic differentiation features.



In [1]:
from jcm.model import Model
from jcm.terrain import TerrainData
from jcm.forcing import ForcingData
from importlib import resources
from jcm.physics.speedy.speedy_coords import get_speedy_coords

# Load the terrain and forcing data (interpolated to the default model grid)
data_dir = resources.files('jcm.data.bc.t30.clim')
coords = get_speedy_coords()
realistic_terrain = TerrainData.from_file(data_dir / 'terrain.nc', coords=coords)
realistic_forcing = ForcingData.from_file(data_dir / 'forcing.nc', coords=coords)

W0000 00:00:1770312924.602024 8412066 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1770312924.618202 8412066 service.cc:145] XLA service 0x30a10a240 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770312924.618211 8412066 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1770312924.619329 8412066 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1770312924.619336 8412066 mps_client.cc:384] XLA backend will use up to 28990554112 bytes on device 0 for SimpleAllocator.


Metal device set to: Apple M3 Max

systemMemory: 36.00 GB
maxCacheSize: 13.50 GB



/Users/ellend/Documents/Development/JAXathon/jax-gcm/jcm/data/bc/interpolate.py:47: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge([daily_time_vars, ds_monthly[non_time_vars]])


In [2]:
model = Model(
    coords=coords,
    terrain=realistic_terrain,
)

In [3]:
predictions = model.run(
    save_interval=5,
    total_time=30,
    output_averages=True,
    forcing=realistic_forcing,
)

In [4]:
pred_ds = predictions.to_xarray()
pred_ds

<xarray.Dataset> Size: 22MB
Dimensions:                          (time: 6, level: 8, lon: 96, lat: 48,
                                      lev_edge: 9, id: 2)
Coordinates:
  * time                             (time) datetime64[ns] 48B 2000-01-01 ......
  * level                            (level) float32 32B 0.95 0.835 ... 0.025
  * lon                              (lon) float64 768B 0.0 3.75 ... 352.5 356.2
  * lat                              (lat) float64 384B -87.16 -83.48 ... 87.16
  * lev_edge                         (lev_edge) int32 36B 0 1 2 3 4 5 6 7 8
  * id                               (id) int32 8B 1 2
Data variables: (12/89)
    v_wind                           (time, level, lon, lat) float32 885kB ...
    shortwave_rad.icltop             (time, lon, lat) float32 111kB ...
    convection.cbmf                  (time, lon, lat) float32 111kB ...
    mod_radcon.tau2.1                (time, level, lon, lat) float32 885kB ...
    mod_radcon.flux.2                (time, lon, lat) float32 111kB ...
    surface_flux.ustr.2              (time, lon, lat) float32 111kB ...
    ...                               ...
    u_wind                           (time, level, lon, lat) float32 885kB ...
    mod_radcon.stratc.0              (time, lon, lat) float32 111kB ...
    shortwave_rad.ftop               (time, lon, lat) float32 111kB ...
    mod_radcon.st4a.1                (time, level, lon, lat) float32 885kB ...
    humidity.qsat                    (time, level, lon, lat) float32 885kB ...
    condensation.dtlsc               (time, level, lon, lat) float32 885kB ...

In [ ]:
pred_ds['condensation.precls'].plot(x='lon', y='lat', col='time', col_wrap=2, aspect=2)
pred_ds['convection.precnv'].plot(x='lon', y='lat', col='time', col_wrap=2, aspect=2)

In [ ]:
pred_ds['specific_humidity'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=3, aspect=6, yincrease=False)
pred_ds['specific_humidity'].isel(level=3).plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

### Clouds!

In [ ]:
pred_ds['shortwave_rad.cloudc'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)
pred_ds['shortwave_rad.qcloud'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

## Continue running

Continue a simulation from the previous state for an additional 30 days, saving every 5 days and outputting averages.


In [ ]:
predictions = model.resume(
    total_time=30,
    output_averages=True,
    forcing=realistic_forcing,
)

Sometimes, for example in a coupled run we want a pure JAX interface to enable compilation of a larger model. We can do this using the `run_from_state` method:

```python
model.run_from_state(
    initial_state=model._final_modal_state,
    total_time=30,
    output_averages=True,
    forcing=realistic_forcing,
)
```

## Note on speed and efficiency

Some preliminary tests show very good performance on a variety of devices but especially on GPUs and TPUs. For example, on a V100 GPU we see around 10x speedup compared to a CPU run. JCM also supports parallelization across multiple devices using JAX's `pmap` functionality, which can further enhance performance for large-scale simulations.

<center><img src="jcm_scaling.png" alt="JCM Performance" width="400"/></center>

## Model Gradients

We can compute gradients of model outputs with respect to model parameters using JAX's automatic differentiation. 

In [ ]:
import jax
import jax.numpy as jnp
import jax.tree_util as jtu
from jcm.physics.speedy.speedy_physics import SpeedyPhysics
from jcm.physics.speedy.params import Parameters

# Key step - create a function that makes a model given parameters
def model_run_wrapper(params):
    model = Model(coords=coords, terrain=realistic_terrain, physics=SpeedyPhysics(parameters=params))

    return model.run(save_interval=1, total_time=1, forcing=realistic_forcing, output_averages=True)

In [ ]:
# VJP is the vector-Jacobian product and calculates the gradient of outputs with respect to inputs

primal, vjp_fn = jax.vjp(model_run_wrapper, Parameters.default())

In [ ]:
def create_cotangent(primal):
    cotangent = jtu.tree_map(jnp.zeros_like, primal)
    cotangent.physics.shortwave_rad.ftop = jtu.tree_map(jnp.ones_like, cotangent.physics.shortwave_rad.ftop)
    return cotangent

grad, = vjp_fn(create_cotangent(primal))

In [ ]:
print(grad)

We can also calculate the Jacobian-vector product (JVP) to get gradients of specific outputs with respect to a parameter

In [ ]:
# Define a tangent vector for JVP
params = Parameters.default()
tangent = Parameters.float_zeros()
tangent.mod_radcon.albsea = jnp.array(1.)

y, jvp_at_x = jax.jvp(model_run_wrapper, (params,), (tangent,))

In [ ]:
import xarray as xr
xr.DataArray(jvp_at_x.physics.surface_flux.hfluxn[0, :, :, 0], dims=['lon', 'lat'],
                coords={'lat': pred_ds.lat, 'lon': pred_ds.lon}).plot(x='lon', y='lat', size=3, aspect=2)

## Advanced

Sometimes it can be useful to control the compilation or execution of the JAX code more directly. JAX has configuration functions to facilitate this.

In [ ]:
print("JAX device:", jax.devices())

# import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"  # for use on ARM Macs

jax.config.update('jax_disable_jit', False) # disable if necessary for debugging
jax.config.update("jax_debug_infs", True) # watch for model blowup
jax.config.update("jax_debug_nans", False) # some physics fields might be nan